# R1-C02 | Benchmark TEST V01 — comparaciones sin entrenamiento

**Objeto:** contrastar PI-LSTM V2.4 V1 RAW congelado contra cuatro referencias: persistencia instantánea (`LAST`), persistencia diaria (`DAILY48`), persistencia del índice de cielo despejado (`CS_INDEX48`) y climatología histórica (`CLIM_TRAIN`).

**Tres dominios de reporte distintos:** ventanas superpuestas (`OVERLAP`, 17 497×24), por lead de 1 a 24 (`BY_LEAD`), y trayectoria única `non_overlapping_blocks` (`NOB`, 35 016 registros). Se evalúa GLOBAL y DAYTIME en cada caso. Los errores de energía **solo** se calculan para NOB. Los benchmarks no usan meteorología futura observada. No se modifica ni se reentrena el modelo.

**Información disponible diferente:** la predicción PI-LSTM de TEST es condicional a meteorología futura observada; los benchmarks solo emplean irradiancia histórica disponible y geometría/clear-sky. Este contraste **no es una comparación operativa con igualdad de predictores**.

Archivos en `/content`: histórico NSRDB, `.keras` V2.4 V1, ambos `.joblib`, `model_metadata_v24_v1.json`, `final_canonical_metrics_v24_v1_raw.csv` y, de preferencia, `test_chronological_non_overlapping_v24_v1.csv` para cotejar NOB. No cargar ninguna versión calibrada isotónicamente.

In [ ]:
from pathlib import Path
import os,hashlib,glob,json
ROOT=Path('/content') if Path('/content').exists() else Path('/mnt/data')
os.chdir(ROOT)
def locate(stem,ext,required=True):
    matches=[p for p in ROOT.iterdir() if p.is_file() and p.suffix.lower()==ext and (p.stem==stem or p.stem.startswith(stem+'('))]
    exact=[p for p in matches if p.stem==stem]
    if exact: return exact[0]
    if len(matches)==1: return matches[0]
    if not required and not matches:return None
    raise FileNotFoundError(f'Archivo no localizado inequívocamente: {stem}{ext}; coincidencias: {[p.name for p in matches]}')
def sha(p):
 h=hashlib.sha256()
 with open(p,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()
DATA=locate('consol_tumbabiro_2026_total','.csv')
MODEL=locate('PI_LSTM_V2_4_V1_logDHI_partition_stride2','.keras')
SC_H=locate('scaler_hist_v24_v1','.joblib')
SC_F=locate('scaler_fut_v24_v1','.joblib')
META=locate('model_metadata_v24_v1','.json')
MODEL_SHA='5aef4783a3dc0b37b98711ebb489d20458f6c5efd2308ca579909e4cc8724a4e'
assert sha(MODEL)==MODEL_SHA,'HASH DE MODELO DIFERENTE AL CANÓNICO'
for p in [DATA,MODEL,SC_H,SC_F,META]:print(p.name, p.stat().st_size)
print('Modelo SHA256 PASS:',sha(MODEL))

CANON_METRICS=locate('final_canonical_metrics_v24_v1_raw','.csv',required=True)
CANON_NOB=locate('test_chronological_non_overlapping_v24_v1','.csv',required=False)
print('Referencia:',CANON_METRICS, 'NOB:',CANON_NOB)

In [ ]:
# ============================================================
# 1. PI-LSTM V2.4 — CONFIGURACIÓN GENERAL
# ============================================================



import os
import gc
import json
import random
import hashlib
from dataclasses import dataclass, asdict
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    LayerNormalization,
    AdditiveAttention,
    TimeDistributed,
    Rescaling,
    Layer
)

from tensorflow.keras.models import Model

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)


# ============================================================
# REPRODUCIBILIDAD
# ============================================================

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


# ============================================================
# CONFIGURACIÓN CANÓNICA PI-LSTM V2.4
# ============================================================

@dataclass(frozen=True)
class CFG:

    # --------------------------------------------------------
    # Archivos
    # --------------------------------------------------------

    data_path: str = "consol_tumbabiro_2026_total.csv"

    model_path: str = (
        "PI_LSTM_V2_4_logDHI_partition_stride2.keras"
    )

    scaler_hist_path: str = (
        "scaler_hist_v24.joblib"
    )

    scaler_fut_path: str = (
        "scaler_fut_v24.joblib"
    )

    metadata_path: str = (
        "model_metadata_v24.json"
    )

    metrics_path: str = (
        "test_metrics_v24.csv"
    )


    # --------------------------------------------------------
    # División temporal
    # --------------------------------------------------------

    train_end: str = "2019-12-31 23:30:00"
    val_end: str = "2022-12-31 23:30:00"


    # --------------------------------------------------------
    # Ventanas temporales
    # --------------------------------------------------------

    L: int = 72
    H: int = 24
    stride: int = 2

    batch_size: int = 32

    epochs: int = 25


    # --------------------------------------------------------
    # Optimización
    # --------------------------------------------------------

    lr: float = 3e-4

    patience_es: int = 8

    patience_rlr: int = 3

    reduce_lr_factor: float = 0.5

    min_lr: float = 1e-5


    # --------------------------------------------------------
    # Arquitectura
    # --------------------------------------------------------

    latent_enc: int = 96
    latent_dec: int = 96

    dropout: float = 0.10


    # --------------------------------------------------------
    # Umbrales físicos
    # --------------------------------------------------------

    thr_cs: float = 20.0

    thr_day_cosz: float = 0.05


    # --------------------------------------------------------
    # Límites de las salidas físicas
    # --------------------------------------------------------

    # Rama directa:
    # la auditoría histórica mostró max(kDNI) = 1.0
    kdni_max: float = 1.20

    # Rama difusa:
    # auditoría histórica sin clipping:
    # max(kDHI) = 7.918
    #
    # Se adopta 8.0 como límite físico-numérico superior.
    kdhi_max: float = 8.00


    # --------------------------------------------------------
    # Pesos de la función de pérdida
    # --------------------------------------------------------

    lambda_kdni: float = 0.40

    # En V2.4 esta ponderación se aplica a qDHI = log1p(kDHI)
    lambda_qdhi: float = 1.45

    lambda_ghi: float = 0.65

    lambda_ratio: float = 1.90

    lambda_energy: float = 0.27


# ============================================================
# INSTANCIAR CONFIGURACIÓN
# ============================================================

cfg = CFG()


# ============================================================
# PARÁMETROS DERIVADOS DE LA TRANSFORMACIÓN DIFUSA
# ============================================================

# qDHI = log(1 + kDHI)
#
# Si kDHI_max = 8:
#
# qDHI_max = log(9) ≈ 2.1972246

QDHI_MAX = float(np.log1p(cfg.kdhi_max))


# ============================================================
# RESUMEN DE CONFIGURACIÓN
# ============================================================

print("TensorFlow:", tf.__version__)

print("\n=== PI-LSTM V2.4 ===")

print("SEED          :", SEED)

print("L             :", cfg.L)
print("H             :", cfg.H)
print("stride        :", cfg.stride)

print("batch_size    :", cfg.batch_size)
print("epochs_max    :", cfg.epochs)

print("learning rate :", cfg.lr)

print("kDNI max      :", cfg.kdni_max)
print("kDHI max      :", cfg.kdhi_max)

print("qDHI max      :", QDHI_MAX)

print("\nArtefactos:")
print("  modelo      :", cfg.model_path)
print("  scaler hist :", cfg.scaler_hist_path)
print("  scaler fut  :", cfg.scaler_fut_path)
print("  metadata    :", cfg.metadata_path)
print("  métricas    :", cfg.metrics_path)

cfg

# ============================================================
# VERIFICACIÓN DEL ENTORNO DE CÓMPUTO
# ============================================================

print("\n=== DISPOSITIVOS DISPONIBLES ===")

print(
    "GPU detectadas:",
    tf.config.list_physical_devices("GPU")
)

In [ ]:
object.__setattr__(cfg,'data_path',str(DATA)); object.__setattr__(cfg,'model_path',str(MODEL))
print('Datos:',cfg.data_path,'Modelo:',cfg.model_path)


In [ ]:
# ============================================================
# 2. PI-LSTM V2.4 — COLUMNAS CANÓNICAS Y CARGA DEL HISTÓRICO
# ============================================================

# ------------------------------------------------------------
# Columnas temporales
# ------------------------------------------------------------

COL_DATE = "Date"
COL_HOUR = "Hour"
COL_MIN  = "Minute"


# ------------------------------------------------------------
# Variables meteorológicas
# ------------------------------------------------------------

COL_TEMP = "Temperature"
COL_DEW  = "Dew Point"
COL_RH   = "Relative Humidity"
COL_PRES = "Pressure"
COL_AOD  = "Aerosol Optical Depth"
COL_PW   = "Precipitable Water"
COL_WSPD = "Wind Speed"
COL_WDIR = "Wind Direction"


# ------------------------------------------------------------
# Variables radiativas
# ------------------------------------------------------------

COL_GHI = "GHI"
COL_DNI = "DNI"
COL_DHI = "DHI"

COL_THZ = "Solar Zenith Angle"

COL_CSGHI = "Clearsky GHI"
COL_CSDNI = "Clearsky DNI"
COL_CSDHI = "Clearsky DHI"


# ============================================================
# CONTRATO MÍNIMO DEL ARCHIVO HISTÓRICO
# ============================================================

REQUIRED_RAW_COLS = [
    COL_DATE,
    COL_HOUR,
    COL_MIN,

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,
    COL_WDIR,

    COL_GHI,
    COL_DNI,
    COL_DHI,
    COL_THZ,

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# VERIFICACIÓN DE RUTA
# ============================================================

data_path = Path(cfg.data_path)

print("Directorio de trabajo:")
print(" ", Path.cwd())

print("\nArchivo histórico esperado:")
print(" ", data_path.resolve())


if not data_path.exists():

    raise FileNotFoundError(
        "\nNo se encontró el archivo histórico requerido:\n"
        f"  {data_path.resolve()}\n\n"
        "Verifique que 'consol_tumbabiro_2026_total.csv' "
        "se encuentre en el directorio de trabajo actual "
        "o modifique cfg.data_path con la ruta correcta."
    )


# ============================================================
# CARGA DEL CSV
# ============================================================

df_raw = pd.read_csv(
    data_path,
    low_memory=False
)


# ============================================================
# AUDITORÍA BÁSICA DE ESTRUCTURA
# ============================================================

missing_raw_cols = [
    c
    for c in REQUIRED_RAW_COLS
    if c not in df_raw.columns
]

if missing_raw_cols:

    raise ValueError(
        "Faltan columnas obligatorias en el histórico:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_raw_cols
        )
    )


# ============================================================
# RESUMEN
# ============================================================

file_size_mb = (
    data_path.stat().st_size
    / (1024 ** 2)
)

print("\n✓ Archivo histórico encontrado")
print(
    f"✓ Tamaño archivo : {file_size_mb:.2f} MB"
)

print(
    f"✓ Dimensión raw  : {df_raw.shape}"
)

print(
    f"✓ Número columnas: {df_raw.shape[1]}"
)

print(
    f"✓ Columnas contractuales presentes: "
    f"{len(REQUIRED_RAW_COLS)} / {len(REQUIRED_RAW_COLS)}"
)

print("\nPrimeras columnas:")
print(
    df_raw.columns.tolist()
)

In [ ]:
# ============================================================
# 3. PI-LSTM V2.4 — TIMESTAMP E INGENIERÍA FÍSICA BÁSICA
# ============================================================


# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

# ============================================================
# 1. CONSTRUCCIÓN ROBUSTA DEL TIMESTAMP
# ============================================================

def build_timestamp(df):
    """
    Construye timestamp a partir de Date, Hour y Minute.

    El histórico consolidado utiliza Date en formato:
        DD/MM/YYYY

    Ejemplo:
        13/01/1998 = 13 de enero de 1998

    Se utiliza un formato explícito para evitar interpretaciones
    ambiguas de pandas.
    """

    out = df.copy()

    # --------------------------------------------------------
    # Fecha: formato explícito DD/MM/YYYY
    # --------------------------------------------------------

    date_raw = (
        out[COL_DATE]
        .astype(str)
        .str.strip()
    )

    date_parsed = pd.to_datetime(
        date_raw,
        format="%d/%m/%Y",
        errors="coerce"
    )

    # --------------------------------------------------------
    # Hora y minuto
    # --------------------------------------------------------

    hour = pd.to_numeric(
        out[COL_HOUR],
        errors="coerce"
    )

    minute = pd.to_numeric(
        out[COL_MIN],
        errors="coerce"
    )

    # --------------------------------------------------------
    # Validaciones de hora y minuto
    # --------------------------------------------------------

    invalid_hour = (
        hour.isna()
        | (hour < 0)
        | (hour > 23)
    )

    invalid_minute = (
        minute.isna()
        | ~minute.isin([0, 30])
    )

    if invalid_hour.any():

        print(
            "\nPrimeros valores inválidos de Hour:"
        )

        print(
            out.loc[
                invalid_hour,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Hour contiene valores inválidos."
        )

    if invalid_minute.any():

        print(
            "\nPrimeros valores inválidos de Minute:"
        )

        print(
            out.loc[
                invalid_minute,
                [COL_DATE, COL_HOUR, COL_MIN]
            ]
            .head(20)
            .to_string(index=False)
        )

        raise ValueError(
            "La columna Minute contiene valores distintos "
            "de 0 o 30."
        )

    # --------------------------------------------------------
    # Timestamp final
    # --------------------------------------------------------

    out["timestamp"] = (
        date_parsed
        + pd.to_timedelta(
            hour,
            unit="h"
        )
        + pd.to_timedelta(
            minute,
            unit="m"
        )
    )

    return out


# ============================================================
# 2. FEATURES TEMPORALES Y GEOMETRÍA
# ============================================================

def add_time_features(df):

    out = df.copy()

    ts = out["timestamp"]

    # --------------------------------------------------------
    # Ciclo diario
    # --------------------------------------------------------

    hour_float = (
        ts.dt.hour
        + ts.dt.minute / 60.0
    )

    out["sin_hour"] = np.sin(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)

    out["cos_hour"] = np.cos(
        2.0 * np.pi * hour_float / 24.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # Ciclo anual
    # Se conserva la formulación V2.3: 365.25 días
    # --------------------------------------------------------

    doy = ts.dt.dayofyear.astype(float)

    out["sin_doy"] = np.sin(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)

    out["cos_doy"] = np.cos(
        2.0 * np.pi * doy / 365.25
    ).astype(np.float32)


    # --------------------------------------------------------
    # Coseno cenital positivo
    # --------------------------------------------------------

    theta = np.deg2rad(
        pd.to_numeric(
            out[COL_THZ],
            errors="coerce"
        ).to_numpy(dtype=np.float32)
    )

    out["cosz_plus"] = np.maximum(
        np.cos(theta),
        0.0
    ).astype(np.float32)

    return out


# ============================================================
# 3. ÍNDICES CLEAR-SKY — V2.4
# ============================================================

def add_kspace_features(df, thr_cs):

    out = df.copy()

    # --------------------------------------------------------
    # Convertir a arrays numéricos
    # --------------------------------------------------------

    csghi = pd.to_numeric(
        out[COL_CSGHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdni = pd.to_numeric(
        out[COL_CSDNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    csdhi = pd.to_numeric(
        out[COL_CSDHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    ghi = pd.to_numeric(
        out[COL_GHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dni = pd.to_numeric(
        out[COL_DNI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)

    dhi = pd.to_numeric(
        out[COL_DHI],
        errors="coerce"
    ).to_numpy(dtype=np.float32)


    # --------------------------------------------------------
    # kGHI
    # Se conserva como feature auxiliar.
    # --------------------------------------------------------

    out["kGHI"] = np.where(
        csghi > thr_cs,
        ghi / csghi,
        0.0
    ).astype(np.float32)

    out["kGHI"] = np.clip(
        out["kGHI"],
        0.0,
        2.0
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDNI
    # Auditoría histórica:
    # max observado = 1.0
    # límite de salida = 1.20
    # --------------------------------------------------------

    out["kDNI"] = np.where(
        csdni > thr_cs,
        dni / csdni,
        0.0
    ).astype(np.float32)

    # Protección numérica histórica.
    # No afecta los datos auditados.
    out["kDNI"] = np.clip(
        out["kDNI"],
        0.0,
        cfg.kdni_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # kDHI RAW
    #
    # V2.4 elimina el antiguo clipping a 2.0.
    # La auditoría histórica mostró:
    #
    # max = 7.918
    # P95 = 4.778
    # P99 = 5.918
    #
    # kdhi_max = 8.0 actúa únicamente como protección
    # físico-numérica y no censura el histórico auditado.
    # --------------------------------------------------------

    kdhi_raw = np.where(
        csdhi > thr_cs,
        dhi / csdhi,
        0.0
    ).astype(np.float32)

    out["kDHI"] = np.clip(
        kdhi_raw,
        0.0,
        cfg.kdhi_max
    ).astype(np.float32)


    # --------------------------------------------------------
    # Transformación logarítmica difusa
    #
    # qDHI = log(1 + kDHI)
    # --------------------------------------------------------

    out["qDHI"] = np.log1p(
        out["kDHI"].to_numpy(dtype=np.float32)
    ).astype(np.float32)


    return out


# ============================================================
# 4. VARIABLES DE PARTICIÓN, DINÁMICA Y MEMORIA
# ============================================================

def add_partition_features(df, thr_cs):

    out = df.copy()

    eps = 1e-6


    # --------------------------------------------------------
    # Rampas radiativas
    # --------------------------------------------------------

    out["dGHI"] = (
        out[COL_GHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDNI"] = (
        out[COL_DNI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )

    out["dDHI"] = (
        out[COL_DHI]
        .diff()
        .fillna(0.0)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción difusa
    # --------------------------------------------------------

    out["diffuse_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        out[COL_DHI]
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["diffuse_ratio"] = (
        out["diffuse_ratio"]
        .clip(0.0, 1.2)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Fracción directa horizontal
    # --------------------------------------------------------

    out["beam_ratio"] = np.where(
        out[COL_GHI] > thr_cs,
        (
            out[COL_DNI]
            * out["cosz_plus"]
        )
        / (out[COL_GHI] + eps),
        0.0
    ).astype(np.float32)

    out["beam_ratio"] = (
        out["beam_ratio"]
        .clip(0.0, 1.5)
        .astype(np.float32)
    )


    # --------------------------------------------------------
    # Variables con lag 1
    #
    # Se conserva kDHI como feature física histórica.
    # Se añade qDHI para disponer también de memoria en
    # el espacio transformado.
    # --------------------------------------------------------

    lag_bases = [
        COL_DNI,
        COL_DHI,
        COL_GHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in lag_bases:

        out[f"{base}_lag1"] = (
            out[base]
            .shift(1)
        )


    # --------------------------------------------------------
    # Promedios móviles de tres pasos previos
    # 3 × 30 min = 90 min
    # --------------------------------------------------------

    roll_bases = [
        COL_DNI,
        COL_DHI,

        "kGHI",
        "kDNI",
        "kDHI",
        "qDHI",

        "diffuse_ratio",
        "beam_ratio",
    ]

    for base in roll_bases:

        out[f"{base}_roll3"] = (
            out[base]
            .shift(1)
            .rolling(
                window=3,
                min_periods=1
            )
            .mean()
        )


    # --------------------------------------------------------
    # Dirección del viento como variable circular
    # --------------------------------------------------------

    wdir = np.deg2rad(
        pd.to_numeric(
            out[COL_WDIR],
            errors="coerce"
        ).fillna(0.0)
    )

    out["sin_wdir"] = np.sin(
        wdir
    ).astype(np.float32)

    out["cos_wdir"] = np.cos(
        wdir
    ).astype(np.float32)


    # --------------------------------------------------------
    # En esta etapa únicamente los NaN producidos por lag/roll
    # iniciales se sustituyen por cero, manteniendo la lógica
    # original de V2.3.
    # --------------------------------------------------------

    return out.fillna(0.0)


# ============================================================
# 5. CONSTRUIR DATAFRAME TEMPORAL
# ============================================================

df = build_timestamp(df_raw)


# ------------------------------------------------------------
# Auditoría del timestamp antes de eliminar registros
# ------------------------------------------------------------

n_bad_timestamp = int(
    df["timestamp"].isna().sum()
)

print(
    "Timestamps inválidos:",
    n_bad_timestamp
)

if n_bad_timestamp > 0:

    print(
        "\nPrimeros registros con timestamp inválido:"
    )

    print(
        df.loc[
            df["timestamp"].isna(),
            [COL_DATE, COL_HOUR, COL_MIN]
        ]
        .head(20)
        .to_string(index=False)
    )

    raise ValueError(
        "Existen timestamps inválidos. "
        "Debe revisarse el formato de Date antes de continuar."
    )


# ------------------------------------------------------------
# Orden y unicidad
# ------------------------------------------------------------

df = (
    df
    .sort_values("timestamp")
    .reset_index(drop=True)
)


n_duplicates = int(
    df["timestamp"].duplicated().sum()
)

print(
    "Timestamps duplicados:",
    n_duplicates
)

if n_duplicates > 0:

    duplicated_rows = df.loc[
        df["timestamp"].duplicated(
            keep=False
        ),
        [
            "timestamp",
            COL_DATE,
            COL_HOUR,
            COL_MIN
        ]
    ]

    print(
        duplicated_rows
        .head(30)
        .to_string(index=False)
    )

    raise ValueError(
        "Se encontraron timestamps duplicados. "
        "No se eliminarán automáticamente."
    )


# ============================================================
# 6. AUDITORÍA ESTRICTA DE CONTINUIDAD A 30 MIN
# ============================================================

dt_min = (
    df["timestamp"]
    .diff()
    .dt.total_seconds()
    .div(60.0)
)

bad_dt = (
    dt_min.notna()
    & ~np.isclose(
        dt_min,
        30.0
    )
)

n_bad_dt = int(
    bad_dt.sum()
)

print(
    "Intervalos distintos de 30 min:",
    n_bad_dt
)


if n_bad_dt > 0:

    bad_idx = np.where(
        bad_dt.to_numpy()
    )[0]

    gaps = []

    for i in bad_idx[:50]:

        gaps.append({
            "index": int(i),

            "timestamp_anterior":
                df.loc[
                    i - 1,
                    "timestamp"
                ],

            "timestamp_actual":
                df.loc[
                    i,
                    "timestamp"
                ],

            "delta_min":
                float(
                    dt_min.iloc[i]
                ),
        })

    gaps_df = pd.DataFrame(gaps)

    print(
        "\n=== PRIMEROS INTERVALOS ANÓMALOS ==="
    )

    print(
        gaps_df.to_string(
            index=False
        )
    )

    print(
        "\nDistribución de intervalos:"
    )

    print(
        dt_min
        .dropna()
        .value_counts()
        .sort_index()
        .head(30)
    )

    raise ValueError(
        "El histórico no es continuo a 30 min. "
        "Revise los intervalos mostrados antes de continuar."
    )


print(
    "✓ Histórico continuo a 30 min."
)


# ============================================================
# 7. CONSTRUIR FEATURES
# ============================================================

df = add_time_features(df)

df = add_kspace_features(
    df,
    cfg.thr_cs
)

df = add_partition_features(
    df,
    cfg.thr_cs
)


# ============================================================
# 8. VALIDACIONES NUMÉRICAS BÁSICAS
# ============================================================

required_engineered = [
    "timestamp",
    "cosz_plus",

    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    "kGHI",
    "kDNI",
    "kDHI",
    "qDHI",

    "diffuse_ratio",
    "beam_ratio",

    "sin_wdir",
    "cos_wdir",
]


missing_engineered = [
    c
    for c in required_engineered
    if c not in df.columns
]

if missing_engineered:

    raise ValueError(
        "Faltan variables de ingeniería:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_engineered
        )
    )


# ------------------------------------------------------------
# Comprobaciones del dominio
# ------------------------------------------------------------

if (
    df["kDNI"].min() < 0.0
    or
    df["kDNI"].max()
    > cfg.kdni_max + 1e-6
):
    raise ValueError(
        "kDNI fuera del dominio configurado."
    )


if (
    df["kDHI"].min() < 0.0
    or
    df["kDHI"].max()
    > cfg.kdhi_max + 1e-6
):
    raise ValueError(
        "kDHI fuera del dominio configurado."
    )


if (
    df["qDHI"].min() < 0.0
    or
    df["qDHI"].max()
    > QDHI_MAX + 1e-6
):
    raise ValueError(
        "qDHI fuera del dominio configurado."
    )


# ============================================================
# 9. RESUMEN
# ============================================================

print("\n=== FEATURE DATAFRAME V2.4 ===")

print(
    "Dimensión:",
    df.shape
)

print(
    "Rango temporal:",
    df["timestamp"].min(),
    "->",
    df["timestamp"].max()
)

print(
    "kDNI min/max:",
    float(df["kDNI"].min()),
    float(df["kDNI"].max())
)

print(
    "kDHI min/max:",
    float(df["kDHI"].min()),
    float(df["kDHI"].max())
)

print(
    "qDHI min/max:",
    float(df["qDHI"].min()),
    float(df["qDHI"].max())
)

print(
    "QDHI_MAX teórico:",
    QDHI_MAX
)

print(
    "\n✓ Celda 3 completada correctamente."
)

In [ ]:
# ============================================================
# 4. PI-LSTM V2.4 — CONTRATO DE FEATURES Y TARGETS
# ============================================================


# ============================================================
# 1. VARIABLES HISTÓRICAS
#
# Se conserva el contrato dimensional:
#
#     HIST_COLS = 34
#
# Respecto de V2.3:
#
#   kDHI_lag1   -> qDHI_lag1
#   kDHI_roll3  -> qDHI_roll3
#
# kDHI instantáneo se conserva como predictor físico.
# ============================================================

HIST_COLS = [

    # --------------------------------------------------------
    # Persistencia y dinámica radiativa
    # --------------------------------------------------------

    "DNI_lag1",
    "DNI_roll3",

    "DHI_lag1",
    "DHI_roll3",

    "GHI_lag1",

    "dGHI",
    "dDNI",
    "dDHI",


    # --------------------------------------------------------
    # Índices clear-sky instantáneos
    # --------------------------------------------------------

    "kGHI",
    "kDNI",
    "kDHI",


    # --------------------------------------------------------
    # Memoria de índices clear-sky
    # --------------------------------------------------------

    "kGHI_lag1",
    "kGHI_roll3",

    "kDNI_lag1",
    "kDNI_roll3",

    # V2.4:
    # memoria difusa en espacio logarítmico
    "qDHI_lag1",
    "qDHI_roll3",


    # --------------------------------------------------------
    # Memoria de partición radiativa
    # --------------------------------------------------------

    "diffuse_ratio_lag1",
    "diffuse_ratio_roll3",

    "beam_ratio_lag1",
    "beam_ratio_roll3",


    # --------------------------------------------------------
    # Meteorología
    # --------------------------------------------------------

    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    "sin_wdir",
    "cos_wdir",


    # --------------------------------------------------------
    # Geometría y clear-sky
    # --------------------------------------------------------

    "cosz_plus",

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# ============================================================
# 2. VARIABLES FUTURAS ESCALADAS
# ============================================================

FUT_COLS_SCALED = [

    # Geometría
    "cosz_plus",

    # Tiempo cíclico
    "sin_hour",
    "cos_hour",
    "sin_doy",
    "cos_doy",

    # Meteorología
    COL_TEMP,
    COL_DEW,
    COL_RH,
    COL_PRES,
    COL_AOD,
    COL_PW,
    COL_WSPD,

    # Dirección del viento circular
    "sin_wdir",
    "cos_wdir",
]


# ============================================================
# 3. VARIABLES FUTURAS FÍSICAS SIN ESCALAR
# ============================================================

FUT_COLS_RAW = [

    COL_CSGHI,
    COL_CSDNI,
    COL_CSDHI,
]


# Contrato futuro completo

FUT_COLS = (
    FUT_COLS_SCALED
    + FUT_COLS_RAW
)


# ============================================================
# 4. TARGETS DEL MODELO
#
# V2.4:
#
#   salida 1 -> kDNI
#   salida 2 -> qDHI = log(1 + kDHI)
#
# Además se conservan variables físicas auxiliares en Y_EXT
# para construir la loss físicamente informada.
# ============================================================

TARGET_COLS = [
    "kDNI",
    "qDHI",
]


# ============================================================
# 5. TARGET EXTENDIDO PARA LA LOSS
#
# Orden contractual:
#
# 0 -> kDNI_true
# 1 -> qDHI_true
# 2 -> GHI_true
# 3 -> DNI_cs
# 4 -> DHI_cs
# 5 -> GHI_cs
# 6 -> cosz_plus
# 7 -> DHI_true
#
# DHI_true se incorpora explícitamente en V2.4 para facilitar
# diagnósticos y futuras extensiones de la loss.
# ============================================================

Y_EXT_COLS = [

    "kDNI",
    "qDHI",

    COL_GHI,

    COL_CSDNI,
    COL_CSDHI,
    COL_CSGHI,

    "cosz_plus",

    COL_DHI,
]


# ============================================================
# 6. VALIDACIÓN DEL CONTRATO
# ============================================================

print("=== CONTRATO DE VARIABLES PI-LSTM V2.4 ===")

print(
    "HIST_COLS       :",
    len(HIST_COLS)
)

print(
    "FUT scaled      :",
    len(FUT_COLS_SCALED)
)

print(
    "FUT raw         :",
    len(FUT_COLS_RAW)
)

print(
    "FUT total       :",
    len(FUT_COLS)
)

print(
    "TARGET_COLS     :",
    len(TARGET_COLS)
)

print(
    "Y_EXT_COLS      :",
    len(Y_EXT_COLS)
)


# ------------------------------------------------------------
# Contratos dimensionales
# ------------------------------------------------------------

assert len(HIST_COLS) == 34, (
    f"Se esperaban 34 variables históricas "
    f"y se obtuvieron {len(HIST_COLS)}."
)

assert len(FUT_COLS_SCALED) == 14, (
    f"Se esperaban 14 variables futuras escaladas "
    f"y se obtuvieron {len(FUT_COLS_SCALED)}."
)

assert len(FUT_COLS_RAW) == 3

assert len(FUT_COLS) == 17

assert len(TARGET_COLS) == 2

assert len(Y_EXT_COLS) == 8


# ============================================================
# 7. VERIFICAR EXISTENCIA DE TODAS LAS COLUMNAS
# ============================================================

all_required_model_cols = list(dict.fromkeys(
    HIST_COLS
    + FUT_COLS
    + TARGET_COLS
    + Y_EXT_COLS
))


missing_model_cols = [
    c
    for c in all_required_model_cols
    if c not in df.columns
]


if missing_model_cols:

    raise ValueError(
        "Faltan columnas requeridas por el contrato V2.4:\n"
        + "\n".join(
            f"  - {c}"
            for c in missing_model_cols
        )
    )


# ============================================================
# 8. VERIFICACIÓN DE FINITUD
# ============================================================

nonfinite_report = {}

for col in all_required_model_cols:

    values = pd.to_numeric(
        df[col],
        errors="coerce"
    ).to_numpy(dtype=np.float64)

    n_bad = int(
        np.sum(
            ~np.isfinite(values)
        )
    )

    if n_bad > 0:
        nonfinite_report[col] = n_bad


if nonfinite_report:

    raise ValueError(
        "Se encontraron valores no finitos en variables "
        "del contrato:\n"
        + json.dumps(
            nonfinite_report,
            indent=2
        )
    )


# ============================================================
# 9. VERIFICACIONES ESPECÍFICAS DE V2.4
# ============================================================

assert np.all(
    df["qDHI"].to_numpy(dtype=float)
    >= 0.0
)

assert (
    float(df["qDHI"].max())
    <= QDHI_MAX + 1e-6
)

# Verificar consistencia exacta:
#
# qDHI = log1p(kDHI)

qdhi_reconstructed = np.log1p(
    df["kDHI"].to_numpy(dtype=float)
)

qdhi_error = np.max(
    np.abs(
        qdhi_reconstructed
        - df["qDHI"].to_numpy(dtype=float)
    )
)

print(
    "\nMáximo error qDHI - log1p(kDHI):",
    qdhi_error
)

assert qdhi_error < 1e-5


# ============================================================
# 10. RESUMEN DEL CONTRATO
# ============================================================

print("\n✓ Contrato V2.4 válido")

print(
    "✓ Input histórico :",
    f"{cfg.L} × {len(HIST_COLS)}"
)

print(
    "✓ Input futuro    :",
    f"{cfg.H} × {len(FUT_COLS)}"
)

print(
    "✓ Target neuronal :",
    f"{cfg.H} × {len(TARGET_COLS)}"
)

print(
    "✓ Target extendido:",
    f"{cfg.H} × {len(Y_EXT_COLS)}"
)

print("\nTargets neuronales:")
print(TARGET_COLS)

print("\nMemoria difusa V2.4:")
print(
    [
        "kDHI",
        "qDHI_lag1",
        "qDHI_roll3"
    ]
)

In [ ]:
import joblib
meta=json.loads(META.read_text(encoding='utf-8'))
sh=joblib.load(SC_H);sf=joblib.load(SC_F)
assert sh.n_features_in_==34 and sf.n_features_in_==14
assert list(sh.feature_names_in_)==list(HIST_COLS),'Orden histórico incongruente'
assert list(sf.feature_names_in_)==list(FUT_COLS_SCALED),'Orden futuro incongruente'
assert np.isfinite(sh.scale_).all() and np.isfinite(sf.scale_).all()
assert (sh.scale_>0).all() and (sf.scale_>0).all()
train_end=pd.Timestamp(cfg.train_end);val_end=pd.Timestamp(cfg.val_end)
train_df=df.loc[df.timestamp<=train_end].copy().reset_index(drop=True)
val_df=df.loc[(df.timestamp>train_end)&(df.timestamp<=val_end)].copy().reset_index(drop=True)
test_df=df.loc[df.timestamp>val_end].copy().reset_index(drop=True)
assert (len(train_df),len(val_df),len(test_df))==(385680,52608,35088)
assert (test_df.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
assert (test_df.timestamp.dt.date==pd.Timestamp('2024-02-29').date()).sum()==48
# Transformar con objetos congelados: ¡nunca fit()!
XH=sh.transform(test_df[HIST_COLS]).astype('float32')
XF=np.concatenate([sf.transform(test_df[FUT_COLS_SCALED]).astype('float32'),test_df[FUT_COLS_RAW].to_numpy(dtype='float32')],axis=1)
YT=test_df[Y_EXT_COLS].to_numpy(dtype='float32')
assert XH.shape==(35088,34) and XF.shape==(35088,17) and YT.shape==(35088,8)
N=(len(test_df)-cfg.L-cfg.H)//cfg.stride+1
assert N==17497
print('PASS: TRAIN VAL TEST',len(train_df),len(val_df),len(test_df),'ventanas:',N)


In [ ]:
# Cargar modelo sin compilar ni entrenar; soportar capas serializadas del notebook original.
# Algunas versiones de Keras requieren registrar la capa Expm1Layer.
import tensorflow as tf
from tensorflow.keras.utils import register_keras_serializable
@register_keras_serializable(package='Custom')
class Expm1Layer(tf.keras.layers.Layer):
 def call(self,inputs):return tf.math.expm1(inputs)
 def get_config(self):return super().get_config()
try:
 model=tf.keras.models.load_model(MODEL,compile=False,custom_objects={'Expm1Layer':Expm1Layer},safe_mode=True)
except Exception as exc:
 raise RuntimeError('No se pudo cargar el modelo congelado con seguridad. Comparta el traceback; no desactive safe_mode ni altere el modelo.') from exc
assert model.input_shape[0][-2:]==(72,34) and model.input_shape[1][-2:]==(24,17)
assert model.output_shape[-2:]==(24,2)
assert model.count_params()==110146
print('PASS:',model.input_shape,model.output_shape,model.count_params(),'TF',tf.__version__)


In [ ]:
# Inferencia sobre 17 497 ventanas en el mismo orden canónico, sin crear matrices 3D completas de inputs.
# Se conserva salida windows completa 17 497 x 24 x 2 (~3.4 MB).
from math import ceil
BATCH=64
P=np.empty((N,cfg.H,2),dtype='float32')
for a in range(0,N,BATCH):
 ids=np.arange(a,min(a+BATCH,N));starts=ids*cfg.stride; ends=starts+cfg.L
 hx=np.stack([XH[s:s+cfg.L] for s in starts])
 fx=np.stack([XF[e:e+cfg.H] for e in ends])
 P[a:a+len(ids)]=model([hx,fx],training=False).numpy()
 if a==0 or (a//BATCH)%40==0:print('ventanas procesadas',min(a+BATCH,N),'/',N)
assert np.isfinite(P).all()
assert P[:,:,0].min()>=-1e-5 and P[:,:,0].max()<=1.20001
assert P[:,:,1].min()>=-1e-5 and P[:,:,1].max()<=np.log1p(8)+1e-5
print('PASS inferencia congelada:',P.shape)


In [ ]:
# Reconstrucción física en cada ventana (evaluación de forecasting por lead).
ids=np.arange(N);starts=ids*cfg.stride; ends=starts+cfg.L
aux=np.stack([YT[e:e+cfg.H] for e in ends])
cosz=aux[:,:,6];dni_cs=aux[:,:,3];dhi_cs=aux[:,:,4]
pdni=P[:,:,0]*dni_cs;pdhi=np.expm1(P[:,:,1])*dhi_cs;pghi=pdni*cosz+pdhi
obsghi=aux[:,:,2];obsdhi=aux[:,:,7];obsdni=test_df['DNI'].to_numpy(dtype='float32')[ends[:,None]+np.arange(cfg.H)]
assert np.max(np.abs(pghi-(pdni*cosz+pdhi)))<1e-3
assert np.isfinite(pghi).all()
from sklearn.metrics import mean_squared_error,mean_absolute_error,r2_score
def stats(obs,pred):
 obs=np.asarray(obs).ravel();pred=np.asarray(pred).ravel();keep=np.isfinite(obs)&np.isfinite(pred);obs=obs[keep];pred=pred[keep]
 return {'N':len(obs),'RMSE':float(np.sqrt(np.mean((pred-obs)**2))),'MAE':float(np.mean(np.abs(pred-obs))),'Bias':float(np.mean(pred-obs)),'R2':float(r2_score(obs,pred))}
rows=[]
for var,o,p in [('GHI',obsghi,pghi),('DNI',obsdni,pdni),('DHI',obsdhi,pdhi)]:
 for domain,mask in [('GLOBAL',np.ones_like(cosz,dtype=bool)),('DAYTIME',(cosz>cfg.thr_day_cosz)&(aux[:,:,5]>cfg.thr_cs))]:
  rows.append({'variable':var,'domain':domain,**stats(o[mask],p[mask])})
metrics=pd.DataFrame(rows)
print(metrics.to_string(index=False))
by_lead=[]
for j in range(cfg.H):
 for v,o,p in [('GHI',obsghi,pghi),('DNI',obsdni,pdni),('DHI',obsdhi,pdhi)]:by_lead.append({'lead':j+1,'variable':v,**stats(o[:,j],p[:,j])})
lead_df=pd.DataFrame(by_lead)


In [ ]:
# Reconstrucción non_overlapping_blocks: seleccionar ventanas cada H/stride=12.
step=cfg.H//cfg.stride;sel=np.arange(0,N,step)
assert len(sel)==1459, len(sel)


In [ ]:
idx=ends[sel,None]+np.arange(cfg.H)
timestamps=test_df.timestamp.to_numpy()[idx].reshape(-1)
assert len(timestamps)==35016
assert len(np.unique(timestamps))==len(timestamps)
assert (pd.Series(timestamps).diff().dropna()==pd.Timedelta(minutes=30)).all()
nob=pd.DataFrame({'timestamp':timestamps,'observed_GHI':obsghi[sel].ravel(),'pred_GHI':pghi[sel].ravel(),'observed_DNI':obsdni[sel].ravel(),'pred_DNI':pdni[sel].ravel(),'observed_DHI':obsdhi[sel].ravel(),'pred_DHI':pdhi[sel].ravel()})
print('PASS NOB:',len(nob),nob.timestamp.min(),nob.timestamp.max())
print('NOB energía [kWh/m2] (observado,predicho,error_pct):')
energy=[]
for v in ['GHI','DNI','DHI']:
 o=nob['observed_'+v].sum()*0.5/1000;p=nob['pred_'+v].sum()*0.5/1000
 energy.append({'variable':v,'observed_kWh_m2':o,'predicted_kWh_m2':p,'error_pct':100*(p/o-1)})
energy=pd.DataFrame(energy);print(energy.to_string(index=False))


## Referencia PI-LSTM — cierre de reproducibilidad NOB

No comparar RMSE de ventanas superpuestas con la referencia oficial, que usa NOB. Se rechaza la ejecución si la NOB actual no concuerda con la canónica (tol. 0.02 W/m² para métricas; discrepancias por redondeo de CSV).

In [ ]:
ref=pd.read_csv(CANON_METRICS).rename(columns={'scope':'domain','RMSE_Wm2':'RMSE','MAE_Wm2':'MAE','Bias_Wm2':'Bias'})
ref['domain']=ref['domain'].str.upper()
print('Columnas canónicas:',list(ref.columns));print(ref.to_string(index=False))
canon_json=json.loads(META.read_text(encoding='utf-8'))
# Comparación de métricas NOB a las tablas canónicas (no a métricas OVERLAP)
nob_rows=[]
for var in ('GHI','DNI','DHI'):
    for domain,mask in [('GLOBAL',np.ones(len(nob),bool)), ('DAYTIME', None)]:
        if mask is None: # índice de la reconstrucción NOB
            m=(cosz[sel].reshape(-1)>cfg.thr_day_cosz)
        else:m=mask
        nob_rows.append({'variable':var,'domain':domain,**stats(nob['observed_'+var].to_numpy()[m],nob['pred_'+var].to_numpy()[m])})
nob_pil=pd.DataFrame(nob_rows)
print('Métricas PI-LSTM NOB:',nob_pil.to_string(index=False))
if {'variable','domain','RMSE'}.issubset(ref.columns):
    merged=nob_pil.merge(ref,on=['variable','domain'],suffixes=('_now','_canon'),validate='1:1')
    assert len(merged)==6
    for col in ['RMSE','MAE','Bias','R2']:
        if col+'_canon' in merged:
            diff=(merged[col+'_now']-merged[col+'_canon']).abs().max()
            print('Máxima diferencia canónica',col,diff)
            assert diff<(0.02 if col!='R2' else 1e-4), f'No conciliado {col}: {diff}'
else:
    print('ATENCIÓN: esquema inesperado de referencia; requiere reconciliación manual antes de afirmar PASS')
if CANON_NOB:
    r=pd.read_csv(CANON_NOB,parse_dates=['timestamp'])
    assert len(r)==len(nob) and np.array_equal(r.timestamp.to_numpy(),pd.to_datetime(nob.timestamp).to_numpy())
    for v in ('GHI','DNI','DHI'):
        discrepancy=np.max(np.abs(r[f'{v}_pred'].to_numpy()-nob[f'pred_{v}'].to_numpy()))
        print('Máxima discrepancia NOB',v,discrepancy)
        assert discrepancy<0.02
    print('PASS: predicciones NOB coinciden con la serie original')

## Definiciones preestablecidas de benchmarks

- **LAST:** cada horizonte de 30 min mantiene el último valor observado al emitir el pronóstico (índice `hist_end−1`), sin consultar valores objetivo del bloque futuro.
- **DAILY48:** se usa el valor observado 48 intervalos antes de cada instante futuro. Dado `H=24` y `L=72`, todas las referencias están disponibles antes de comenzar la predicción. No se usa el dato observado del mismo día futuro.
- **CS_INDEX48:** calcula por componente el cociente `observado(t−48)/clear_sky(t−48)` solo si el clear-sky de origen supera 20 W/m² y reproduce multiplicando el clear-sky del instante objetivo. Denominador bajo → índice cero, además de noche física. Los índices quedan acotados `k_GHI∈[0,1.2]`, `k_DNI∈[0,1.2]`, `k_DHI∈[0,8]` y se registra explícitamente el efecto de estos límites. Este baseline **no fuerza cierre GHI=DNI·cosz+DHI** y no debe presentarse como física-informada comparable.
- **CLIM_TRAIN:** mediana de GHI, DNI, DHI de TRAIN 1998–2019 por día calendario y ranura de 30 min. El 29-feb de 2024 usa la media de las medianas del 28-feb y 1-mar para la misma ranura. Sin datos VAL/TEST. La climatología usa calendario, no meteorología futura observada.

Para reducir ambigüedades, **LAST no incorpora proyección de clear-sky**; como referencia ingenua puede tener irradiancia nocturna y así se evalúa (no se corrige selectivamente). En los otros baselines se preserva el tratamiento de noche definido arriba. Todas las referencias se calculan con el mismo conjunto de objetivos y ventanas de TEST.

In [ ]:
# Integridad cronológica y características físicas a partir del df original, no de las predicciones
assert (df.timestamp.diff().dropna()==pd.Timedelta(minutes=30)).all()
ORIG_COLS={'GHI':COL_GHI,'DNI':COL_DNI,'DHI':COL_DHI}
CLEAR_COLS={'GHI':COL_CSGHI,'DNI':COL_CSDNI,'DHI':COL_CSDHI}
VARS=('GHI','DNI','DHI')
# Variables de entrada para benchmark en un mismo orden 
OBS=np.stack([test_df[ORIG_COLS[v]].to_numpy(dtype='float32') for v in VARS],axis=-1)
CS=np.stack([test_df[CLEAR_COLS[v]].to_numpy(dtype='float32') for v in VARS],axis=-1)
assert OBS.shape==(35088,3) and CS.shape==(35088,3)
assert np.isfinite(OBS).all() and np.isfinite(CS).all()
origins=(np.arange(N)*cfg.stride+cfg.L).astype('int64')
leads=np.arange(cfg.H,dtype='int64')
target_ix=origins[:,None]+leads[None,:]
assert target_ix.shape==(17497,24)
assert np.all(target_ix-origins[:,None]>=0)
# En particular para DAILY48, todo índice usado precede a la emisión
assert np.all(target_ix-48<origins[:,None]) and np.min(target_ix-48)>=0
# Congelar climatología en TRAIN exclusivamente
tr=train_df[['timestamp']+[ORIG_COLS[v] for v in VARS]].copy()
tr['md']=tr.timestamp.dt.strftime('%m-%d')
tr['slot']=tr.timestamp.dt.hour*2+tr.timestamp.dt.minute//30
profile=tr.groupby(['md','slot'],sort=True)[[ORIG_COLS[v] for v in VARS]].median()
assert len(profile)==366*48
print('PASS: bases causales y climatología TRAIN: ', len(profile),'perfiles')

In [ ]:
# Matrices de referencia: N_windows x H x 3, memoria aprox 5 MB por baseline
BENCH={}
BENCH['LAST']=np.broadcast_to(OBS[origins-1,None,:],(N,cfg.H,3)).copy()
BENCH['DAILY48']=OBS[target_ix-48].copy()
source_cs=CS[target_ix-48]
future_cs=CS[target_ix]
ratio=np.divide(OBS[target_ix-48],source_cs,out=np.zeros_like(source_cs),where=source_cs>cfg.thr_cs)
ratio=np.clip(ratio,0,np.array([1.2,1.2,8.0],dtype='float32'))
BENCH['CS_INDEX48']=np.maximum(ratio*future_cs,0)
# Si no hay cielo despejado físico (>0), se impone cero; alternativa explícita, igual a GHI/DNI/DHI clear-sky.
BENCH['CS_INDEX48'][future_cs<=0]=0
# climatología 366 x 48, independientemente del año objetivo
keys=test_df.timestamp.dt.strftime('%m-%d')
slots=(test_df.timestamp.dt.hour*2+test_df.timestamp.dt.minute//30).to_numpy()
clim=np.empty((len(test_df),3),dtype='float32')
for slot in range(48):
    mask=slots==slot
    dates=keys[mask]
    sub=profile.xs(slot,level='slot')
    values=sub.loc[dates].to_numpy(dtype='float32')
    if (dates=='02-29').any():
        leap=(dates=='02-29').to_numpy()
        values[leap]=(sub.loc['02-28'].to_numpy(dtype='float32')+sub.loc['03-01'].to_numpy(dtype='float32'))/2
    clim[mask]=values
BENCH['CLIM_TRAIN']=clim[target_ix].copy()
# Evitar valores negativos en referencias puramente radiativas climatológicas
BENCH['CLIM_TRAIN']=np.maximum(BENCH['CLIM_TRAIN'],0)
# Metadatos QA
for name,pred in BENCH.items():
    assert pred.shape==(N,24,3) and np.isfinite(pred).all(),name
    print(name,'shape',pred.shape,'rango',float(pred.min()),float(pred.max()))
print('No se usaron observaciones futuras ni se reajustaron parámetros del PI-LSTM')

## Comparaciones y exportación

Las observaciones PI-LSTM son exactamente las usadas en la validación congelada; no se recalculan blancos nuevos. La máscara DAYTIME principal replica el notebook original (`cosz_plus>0.05`). Se exporta por separado una sensibilidad al criterio `GHI_cs>20`. Se preservan los nombres y definiciones de las métricas canónicas, con etiquetas explícitas de protocolo.

In [ ]:
# Cubo de observaciones, estimaciones PI-LSTM y masks
TRUTH=np.stack([obsghi,obsdni,obsdhi],axis=-1)
PIL=np.stack([pghi,pdni,pdhi],axis=-1)
assert np.max(np.abs(TRUTH-OBS[target_ix]))<1e-3
BENCH={'PI_LSTM_V24V1':PIL,**BENCH}
M_DAY=(cosz>cfg.thr_day_cosz)
M_DAY_CS=M_DAY&(aux[:,:,5]>cfg.thr_cs)
print('Máscaras DAYTIME diferencia posiciones:',int(np.sum(M_DAY!=M_DAY_CS)))
records=[]; leadrecords=[]; nobrecords=[]; energrecords=[]
for name,pred in BENCH.items():
    for k,v in enumerate(VARS):
        ob=TRUTH[:,:,k];pr=pred[:,:,k]
        for d,m in [('GLOBAL',np.ones_like(M_DAY,bool)),('DAYTIME',M_DAY),('DAYTIME_CS20',M_DAY_CS)]:
            records.append({'model':name,'variable':v,'domain':d,'protocol':'OVERLAP','n_windows':N,**stats(ob[m],pr[m])})
        for j in range(cfg.H):
            for d,m in [('GLOBAL',np.ones(N,bool)),('DAYTIME',M_DAY[:,j])]:
                leadrecords.append({'model':name,'lead':j+1,'variable':v,'domain':d,'protocol':'BY_LEAD',**stats(ob[m,j],pr[m,j])})
        ob_n=ob[sel].reshape(-1); pr_n=pr[sel].reshape(-1)
        for d,m in [('GLOBAL',np.ones(len(ob_n),bool)),('DAYTIME',M_DAY[sel].reshape(-1))]:
            nobrecords.append({'model':name,'variable':v,'domain':d,'protocol':'NOB',**stats(ob_n[m],pr_n[m])})
        eo=float(np.sum(ob_n.astype('float64'))*.5/1000)
        ep=float(np.sum(pr_n.astype('float64'))*.5/1000)
        energrecords.append({'model':name,'variable':v,'n_records':len(ob_n),'observed_kWh_m2':eo,'predicted_kWh_m2':ep,'error_pct':100*(ep/eo-1)})
OVERLAP=pd.DataFrame(records)
BY_LEAD=pd.DataFrame(leadrecords)
NOBMET=pd.DataFrame(nobrecords)
ENERGY=pd.DataFrame(energrecords)
assert len(OVERLAP)==5*3*3 and len(BY_LEAD)==5*24*3*2 and len(NOBMET)==5*3*2
assert len(ENERGY)==5*3
print('OVERLAP:',OVERLAP.shape,'BY_LEAD:',BY_LEAD.shape,'NOB:',NOBMET.shape)
print('\nNOB GLOBAL:');print(NOBMET[NOBMET.domain=='GLOBAL'].to_string(index=False))
print('\nENERGÍA NOB:');print(ENERGY.to_string(index=False))

In [ ]:
# Conciliación cuantitativa OBLIGATORIA de las métricas canónicas NOB
reference=pd.read_csv(CANON_METRICS).rename(columns={'scope':'domain','RMSE_Wm2':'RMSE','MAE_Wm2':'MAE','Bias_Wm2':'Bias'})
reference['domain']=reference['domain'].str.upper()
assert {'variable','domain','RMSE','MAE','Bias','R2'}.issubset(reference.columns)
verify=NOBMET[(NOBMET.model=='PI_LSTM_V24V1')&(NOBMET.domain.isin(['GLOBAL','DAYTIME']))].merge(reference,on=['variable','domain'],suffixes=('_actual','_canonica'),validate='1:1')
assert len(verify)==6
for metric in ('RMSE','MAE','Bias','R2'):
    diff=np.max(np.abs(verify[metric+'_actual']-verify[metric+'_canonica']))
    print(metric,'max delta',diff)
    assert diff<(0.02 if metric!='R2' else 1e-4), f'No conciliado: {metric}: {diff}'
print('PASS NOB: seis filas y cuatro métricas concuerdan con tabla canónica')

In [ ]:
# Exportación controlada. No sobrescribir evidencia previa.
from datetime import datetime, timezone
import zipfile
OUT=ROOT/'R1_C02_TEST_BENCHMARKS_V01'
OUT.mkdir(exist_ok=True)
files={'Benchmark_Overlap.csv':OVERLAP,'Benchmark_ByLead.csv':BY_LEAD,'Benchmark_NOB.csv':NOBMET,'Benchmark_NOB_Energy.csv':ENERGY}
checks={}
for fn,dfout in files.items():
    dest=OUT/fn;dfout.to_csv(dest,index=False,float_format='%.10g')
    checks[fn]=sha(dest)
manifest={'status':'BENCHMARK_EXECUTED_CANONICAL_NOB_CONCILIATED','model_sha256':sha(MODEL),'dataset_sha256':sha(DATA),'scaler_hist_sha256':sha(SC_H),'scaler_fut_sha256':sha(SC_F),'reference_sha256':sha(CANON_METRICS),'tensorflow_version':tf.__version__,'protocols':['OVERLAP','BY_LEAD','NOB'],'reference_models':list(BENCH),'train_climatology_end':'2019-12-31','climatology_feb29':'mean_Feb28_Mar01_mediana_by_slot','irradiance_prediction_reference':'observed future meteorology PI-LSTM only; baseline no future meteorology','daytime_mask':'cosz_plus>0.05; optional DAYTIME_CS20 separately','benchmark_index_caps':{'GHI':1.2,'DNI':1.2,'DHI':8},'no_training':True,'hashes':checks,'warnings':['LAST puede producir irradiancia nocturna; no corregida a posteriori','CS_INDEX48 no impone cierre radiativo','No comparar métricas OVERLAP contra tabla canónica NOB','No es una comparación operativa con igualdad de información futura']}
(OUT/'Manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False),encoding='utf-8')
package=ROOT/'R1_C02_TEST_BENCHMARKS_V01.zip'
with zipfile.ZipFile(package,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.iterdir()):z.write(p,arcname=p.name)
print('ZIP:',package,'SHA256',sha(package));print('Archivos:',sorted(checks))